In [4]:
# -*- coding: utf-8 -*-
"""
Build firm-year innovation score from:
1) stat/graph/main_enriched.parquet
2) stat/公司财务/数据/上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv

Method: Top10Mean of Quality_q within firm-year
- Filter out extreme Quality_q > 1000 at patent level
- Standardize within each year (z-score across firms)

Output (fixed name):
- stat/graph/firm_year_innovation.parquet
"""

from __future__ import annotations
import os
import polars as pl


# -----------------------------
# Config (edit here if needed)
# -----------------------------
PATENT_PATH = r"../graph/main_enriched.parquet"
UCC_LIST_PATH = r"../公司财务/数据/上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv"
OUTPUT_PATH = r"../公司财务/firm_year_innovation.parquet"

TOP_K = 10
QUALITY_CAP = 1000.0


def read_ucc_list(path: str) -> pl.DataFrame:
    """
    Read firm-year UCC list and explode to (Stkid, year, UCC) rows.
    Expected columns (Chinese as in your csv):
      - 证券ID (or stkid)
      - 公司简称 (or shortname)
      - 年份 (or year)
      - 统一社会信用代码列表 (semicolon separated)
    """
    df = pl.read_csv(path, infer_schema_length=10000)

    # Normalize column names (support both Chinese and already-normalized)
    rename_map = {}
    if "证券ID" in df.columns:
        rename_map["证券ID"] = "Stkid"
    if "stkid" in df.columns:
        rename_map["stkid"] = "Stkid"
    if "公司简称" in df.columns:
        rename_map["公司简称"] = "ShortName"
    if "shortname" in df.columns:
        rename_map["shortname"] = "ShortName"
    if "年份" in df.columns:
        rename_map["年份"] = "year"
    if "year" in df.columns:
        rename_map["year"] = "year"
    if "统一社会信用代码列表" in df.columns:
        rename_map["统一社会信用代码列表"] = "UCC_list"
    elif "统一社会信用代码" in df.columns:
        # fallback if someone stored list column under a different name
        rename_map["统一社会信用代码"] = "UCC_list"

    df = df.rename(rename_map)

    required = {"Stkid", "ShortName", "year", "UCC_list"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"[UCC list] Missing required columns: {missing}. Got columns={df.columns}")

    # explode UCC_list -> UCC
    df = (
        df.with_columns(
            pl.col("UCC_list")
            .cast(pl.Utf8)
            .fill_null("")
            .str.strip_chars()
            .str.split(";")
            .alias("UCC_arr")
        )
        .explode("UCC_arr")
        .with_columns(pl.col("UCC_arr").cast(pl.Utf8).str.strip_chars().alias("UCC"))
        .filter(pl.col("UCC").is_not_null() & (pl.col("UCC") != "") & (pl.col("UCC").str.to_lowercase() != "nan"))
        .select(["Stkid", "ShortName", "year", "UCC"])
        .with_columns(pl.col("year").cast(pl.Int32))
        .unique(subset=["Stkid", "year", "UCC"])  # avoid duplicate ucc for same firm-year
    )
    return df


def read_patents(path: str) -> pl.DataFrame:
    """
    Read patent table with needed columns only.
    Expected columns (Chinese as in main_enriched.parquet):
      - 申请年份
      - 统一社会信用代码
      - Quality_q
    """
    # Lazy scan for speed/memory
    lf = (
        pl.scan_parquet(path)
        .select(
            pl.col("申请年份").cast(pl.Int32).alias("year"),
            pl.col("统一社会信用代码")
              .cast(pl.Utf8)
              .str.strip_chars()
              .alias("UCC"),
            pl.col("Quality_q").cast(pl.Float64),
        )
        .filter(pl.col("UCC").is_not_null())
        .filter(pl.col("UCC") != "")
        .filter(pl.col("Quality_q").is_not_null())
        .filter(pl.col("Quality_q") <= QUALITY_CAP)
    )

    # ⬇️ 注意这里，第二个 warning 也一起解决
    return lf.collect(engine="streaming")


def build_innovation_topk_mean(
    patents: pl.DataFrame, ucc_map: pl.DataFrame, top_k: int
) -> pl.DataFrame:
    """
    Join patents to firm-year UCC mapping, then aggregate firm-year innovation.
    Innovation_raw = mean(TopK Quality_q within firm-year)
    PatentCount = count of matched patents within firm-year
    """
    joined = (
        patents.join(ucc_map, on=["UCC", "year"], how="inner")
        .select(["Stkid", "ShortName", "year", "Quality_q"])
    )

    # ----核心：这里就是“算法插拔点”----
    # TopK mean:
    agg = (
        joined.group_by(["Stkid", "ShortName", "year"])
        .agg(
            pl.col("Quality_q").count().alias("PatentCount"),
            pl.col("Quality_q").sort(descending=True).head(top_k).mean().alias("Innovation_raw"),
        )
        .with_columns(pl.lit(f"Top{top_k}Mean").alias("Method"))
    )

    return agg


def add_year_zscore(df: pl.DataFrame) -> pl.DataFrame:
    """
    Compute within-year z-score of Innovation_raw across firms.
    Innovation_z = (raw - mean_y) / std_y
    """
    stats = (
        df.group_by("year")
        .agg(
            pl.col("Innovation_raw").mean().alias("mu"),
            pl.col("Innovation_raw").std().alias("sigma"),
        )
    )
    out = (
        df.join(stats, on="year", how="left")
        .with_columns(
            pl.when((pl.col("sigma").is_null()) | (pl.col("sigma") == 0))
            .then(None)
            .otherwise((pl.col("Innovation_raw") - pl.col("mu")) / pl.col("sigma"))
            .alias("Innovation_z")
        )
        .drop(["mu", "sigma"])
    )
    return out


def main():
    if not os.path.exists(PATENT_PATH):
        raise FileNotFoundError(f"Patent parquet not found: {PATENT_PATH}")
    if not os.path.exists(UCC_LIST_PATH):
        raise FileNotFoundError(f"UCC list csv not found: {UCC_LIST_PATH}")

    print("[1/4] Reading firm-year UCC list...")
    ucc_map = read_ucc_list(UCC_LIST_PATH)
    print(f"  UCC map rows: {ucc_map.height:,}")

    print("[2/4] Reading patents (only needed cols)...")
    patents = read_patents(PATENT_PATH)
    print(f"  Patent rows after filter: {patents.height:,}")

    print("[3/4] Aggregating firm-year innovation (TopK mean)...")
    firm_year = build_innovation_topk_mean(patents, ucc_map, TOP_K)

    # ✅ 过滤 Innovation_raw == 0
    firm_year = firm_year.filter(pl.col("Innovation_raw") > 0)

    print("[4/4] Year-wise z-score standardization...")
    firm_year = add_year_zscore(firm_year)

    # Save output (fixed file name)
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    firm_year.write_parquet(OUTPUT_PATH)
    print(f"Done. Output saved to: {OUTPUT_PATH}")
    print(f"Output rows: {firm_year.height:,}, cols: {firm_year.width}")


if __name__ == "__main__":
    main()


[1/4] Reading firm-year UCC list...
  UCC map rows: 1,182,973
[2/4] Reading patents (only needed cols)...
  Patent rows after filter: 4,323,186
[3/4] Aggregating firm-year innovation (TopK mean)...
[4/4] Year-wise z-score standardization...
Done. Output saved to: ../公司财务/firm_year_innovation.parquet
Output rows: 29,309, cols: 7
